In [0]:
# %sql
# CREATE SCHEMA IF NOT EXISTS dev_ucx.freshdeskwatermark;

In [0]:
# spark.sql("DESCRIBE TABLE hive_metastore.gi_transformed.contacts")

In [0]:
# %sql
# SELECT MAX(CAST(updated_at AS TIMESTAMP))
# FROM dev_ucx.gi_transformed.contacts;

In [0]:
# %sql
# SELECT
#   updated_at,
#   CAST(updated_at AS TIMESTAMP) AS ts
# FROM dev_ucx.gi_transformed.contacts
# WHERE updated_at IS NOT NULL
# LIMIT 20;

In [0]:
# %sql
# CREATE TABLE freshdeskwatermark.contacts(
#   source_name STRING,
#   last_run_time TIMESTAMP
# )
# USING DELTA;

In [0]:
# %sql
# UPDATE freshdeskwatermark.contacts
# SET last_run_time = TIMESTAMP('2026-06-02T12:11:41.000+00:00')
# WHERE source_name = 'freshdesk_contacts'

In [0]:
contacts = spark.read.format("json").option("multiline","true").option("mode","PERMISSIVE").load("/mnt/jspl_datalake/GI_API_RAW_DEV/fresh_desk/contacts/*/*.json")

In [0]:
contacts.write.format("delta") \
    .mode("overwrite") \
    .option("optimizeWrite", "true") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gi_transformed.contacts")

In [0]:
from pyspark.sql.functions import max, col

# Get latest timestamp from Delta table
max_time = spark.sql("""
SELECT MAX(CAST(updated_at AS TIMESTAMP)) AS max_time
FROM gi_transformed.contacts
""").collect()[0][0]

# Update watermark table
if max_time is not None:
    spark.sql(f"""
    UPDATE freshdeskwatermark.contacts
    SET last_run_time = TIMESTAMP('{max_time}')
    WHERE source_name = 'freshdesk_contacts'
    """)
    
    print("Watermark updated to:", max_time)
else:
    print("No data found — watermark not updated")

Watermark updated to: 2026-06-11 05:25:36


In [0]:
%sql
select * from freshdeskwatermark.contacts

source_name,last_run_time
freshdesk_contacts,2026-07-28T11:40:29Z
